In [1]:
!pip install groq -q

import os, json
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)
os.makedirs("agent", exist_ok=True)
print("Ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00
Ready!


In [8]:
treatment_data = {
    "Apple scab": {
        "organic":  ["Apply neem oil spray every 7-10 days",
                     "Use sulfur-based fungicide before infection period",
                     "Remove and destroy all fallen infected leaves"],
        "chemical": ["Apply Captan fungicide at bud break",
                     "Use Myclobutanil (Rally) during growing season",
                     "Spray Mancozeb every 7 days in wet conditions"],
        "prevention": "Plant resistant apple varieties, ensure good air circulation by pruning"
    },
    "Apple Black rot": {
        "organic":  ["Prune and destroy all infected branches and mummified fruit",
                     "Apply copper-based fungicide at bud break",
                     "Remove dead bark and cankers from trees"],
        "chemical": ["Apply Captan or Thiram fungicide during growing season",
                     "Use Myclobutanil for systemic protection",
                     "Spray Ziram at petal fall stage"],
        "prevention": "Remove mummified fruit, prune dead wood, avoid wounding trees"
    },
    "Cedar apple rust": {
        "organic":  ["Apply sulfur-based fungicide before infection period",
                     "Remove nearby juniper or cedar trees if possible",
                     "Use copper spray at early leaf stage"],
        "chemical": ["Apply Myclobutanil (Rally) from pink bud through cover",
                     "Use Propiconazole (Banner Maxx) for systemic control",
                     "Spray Mancozeb as protective treatment"],
        "prevention": "Plant rust-resistant apple varieties, remove nearby juniper hosts"
    },
    "Blueberry healthy": {
        "organic":  ["Continue regular compost application",
                     "Use neem oil as preventive spray monthly",
                     "Maintain proper watering schedule"],
        "chemical": [],
        "prevention": "Regular monitoring, balanced fertilization, proper plant spacing"
    },
    "Cherry Powdery mildew": {
        "organic":  ["Spray diluted baking soda solution (1 tbsp per litre water)",
                     "Apply potassium bicarbonate spray weekly",
                     "Use neem oil every 7 days as preventive"],
        "chemical": ["Apply Myclobutanil (Eagle) at first sign of disease",
                     "Use Trifloxystrobin (Flint) for long-lasting protection",
                     "Spray Tebuconazole every 14 days"],
        "prevention": "Avoid overhead irrigation, improve air circulation by pruning"
    },
    "Cercospora leaf spot Gray leaf spot": {
        "organic":  ["Apply copper hydroxide spray at early infection",
                     "Remove and destroy heavily infected leaves",
                     "Ensure wide plant spacing for airflow"],
        "chemical": ["Apply Propiconazole (Tilt) at tasseling stage",
                     "Use Azoxystrobin + Propiconazole (Quilt) mixture",
                     "Spray Pyraclostrobin (Headline) for best control"],
        "prevention": "Plant resistant corn hybrids, rotate crops annually, till crop debris"
    },
    "Common rust": {
        "organic":  ["Apply sulfur-based fungicide at first sign",
                     "Remove heavily infected leaves to slow spread",
                     "Ensure good air circulation between plants"],
        "chemical": ["Apply Propiconazole (Tilt) at early infection",
                     "Use Azoxystrobin (Quadris) for systemic protection",
                     "Spray Mancozeb as protective treatment in wet weather"],
        "prevention": "Plant rust-resistant corn varieties, avoid late planting"
    },
    "Northern Leaf Blight": {
        "organic":  ["Apply copper hydroxide spray at early infection",
                     "Remove heavily infected leaves promptly",
                     "Ensure proper plant spacing for good airflow"],
        "chemical": ["Apply Propiconazole (Tilt) at tasseling stage",
                     "Use Azoxystrobin + Propiconazole (Quilt) mixture",
                     "Spray Pyraclostrobin (Headline) for best control"],
        "prevention": "Plant resistant corn hybrids, rotate crops annually"
    },
    "Grape Black rot": {
        "organic":  ["Apply copper-based fungicide before bloom",
                     "Remove and destroy mummified berries and infected leaves",
                     "Improve canopy air circulation by pruning"],
        "chemical": ["Apply Myclobutanil (Rally) from early shoot growth",
                     "Use Mancozeb as protective spray every 7-10 days",
                     "Spray Captan after bloom for fruit protection"],
        "prevention": "Remove mummified fruit, prune for airflow, avoid overhead irrigation"
    },
    "Esca (Black Measles)": {
        "organic":  ["Prune infected wood back to healthy tissue",
                     "Seal pruning wounds immediately with wound sealant",
                     "Remove and destroy severely infected vines"],
        "chemical": ["Apply sodium arsenite to cut surfaces (where permitted)",
                     "Use thiophanate-methyl as pruning wound protectant",
                     "Treat with Flusilazole for foliar symptoms"],
        "prevention": "Prune in dry weather, protect wounds immediately, use clean tools"
    },
    "Leaf blight (Isariopsis Leaf Spot)": {
        "organic":  ["Apply copper-based fungicide at first sign",
                     "Remove and destroy infected leaves",
                     "Improve air circulation through canopy management"],
        "chemical": ["Apply Mancozeb every 10-14 days",
                     "Use Myclobutanil for systemic control",
                     "Spray Iprodione at early infection stage"],
        "prevention": "Avoid overhead irrigation, remove infected plant debris after harvest"
    },
    "Haunglongbing (Citrus greening)": {
        "organic":  ["Remove and destroy infected trees immediately",
                     "Use reflective mulch to repel psyllid insects",
                     "Apply kaolin clay spray to deter psyllids"],
        "chemical": ["Apply Imidacloprid (Confidor) to control Asian citrus psyllid",
                     "Use Thiamethoxam (Actara) as systemic insecticide",
                     "Spray Dimethoate for psyllid population control"],
        "prevention": "Use certified disease-free nursery stock, monitor for psyllids regularly"
    },
    "Peach Bacterial spot": {
        "organic":  ["Apply copper-based bactericide at bud swell",
                     "Remove and destroy infected plant material",
                     "Avoid overhead irrigation to reduce leaf wetness"],
        "chemical": ["Apply Oxytetracycline (Mycoshield) during bloom",
                     "Use copper hydroxide sprays through growing season",
                     "Spray Kasugamycin for bacterial control"],
        "prevention": "Plant resistant peach varieties, avoid sites with frequent rain"
    },
    "Pepper Bacterial spot": {
        "organic":  ["Apply copper-based bactericide every 7 days",
                     "Remove infected leaves and plant debris",
                     "Avoid working with plants when wet"],
        "chemical": ["Apply copper hydroxide + Mancozeb mixture",
                     "Use Acibenzolar-S-methyl (Actigard) as resistance inducer",
                     "Spray Kasugamycin at early infection"],
        "prevention": "Use certified disease-free seeds, rotate crops every 2-3 years"
    },
    "Potato Early blight": {
        "organic":  ["Apply neem oil or copper spray weekly",
                     "Mulch around base to prevent soil splash onto leaves",
                     "Remove lower infected leaves immediately"],
        "chemical": ["Apply Chlorothalonil every 7-10 days",
                     "Use Azoxystrobin (Quadris) for systemic protection",
                     "Alternate fungicides to prevent resistance"],
        "prevention": "Crop rotation every 2-3 years, stake plants for airflow, avoid overhead watering"
    },
    "Potato Late blight": {
        "organic":  ["Apply copper-based fungicide immediately at first sign",
                     "Remove and destroy all infected plant parts",
                     "Avoid overhead irrigation — switch to drip"],
        "chemical": ["Apply Chlorothalonil (Daconil) at first sign",
                     "Use Mefenoxam (Ridomil) for soil drenching",
                     "Rotate with Cymoxanil for resistance management"],
        "prevention": "Use certified disease-free seed potatoes, hill soil around stems"
    },
    "Squash Powdery mildew": {
        "organic":  ["Spray diluted baking soda solution (1 tbsp per litre)",
                     "Apply potassium bicarbonate spray weekly",
                     "Use neem oil every 7 days"],
        "chemical": ["Apply Myclobutanil at first sign of disease",
                     "Use Trifloxystrobin for long-lasting protection",
                     "Spray Sulfur fungicide in early morning"],
        "prevention": "Plant resistant varieties, avoid overhead watering, improve spacing"
    },
    "Strawberry Leaf scorch": {
        "organic":  ["Remove and destroy infected leaves",
                     "Apply copper-based fungicide early in season",
                     "Avoid overhead irrigation"],
        "chemical": ["Apply Captan fungicide at first sign",
                     "Use Myclobutanil for systemic control",
                     "Spray Thiram as protective treatment"],
        "prevention": "Use certified disease-free plants, renovate beds after harvest"
    },
    "Tomato Bacterial spot": {
        "organic":  ["Apply copper-based bactericide every 7 days",
                     "Remove infected leaves and avoid working when wet",
                     "Use drip irrigation instead of overhead"],
        "chemical": ["Apply copper hydroxide + Mancozeb mixture",
                     "Use Acibenzolar-S-methyl (Actigard) as resistance inducer",
                     "Spray Kasugamycin at early stages"],
        "prevention": "Use certified disease-free seeds, rotate crops every 2-3 years"
    },
    "Tomato Early blight": {
        "organic":  ["Apply neem oil or copper spray weekly",
                     "Mulch around base to prevent soil splash",
                     "Remove lower infected leaves immediately"],
        "chemical": ["Apply Chlorothalonil every 7-10 days",
                     "Use Azoxystrobin (Quadris) for systemic protection",
                     "Alternate fungicides to prevent resistance"],
        "prevention": "Crop rotation every 2-3 years, stake plants for airflow"
    },
    "Tomato Late blight": {
        "organic":  ["Apply copper-based fungicide immediately",
                     "Remove and destroy all infected plant parts",
                     "Avoid overhead irrigation — use drip instead"],
        "chemical": ["Apply Chlorothalonil (Daconil) at first sign",
                     "Use Mefenoxam (Ridomil) for soil drenching",
                     "Rotate with Cymoxanil for resistance management"],
        "prevention": "Use certified disease-free seeds, avoid wet foliage"
    },
    "Leaf Mold": {
        "organic":  ["Improve greenhouse ventilation immediately",
                     "Remove and destroy infected leaves",
                     "Apply copper-based fungicide weekly"],
        "chemical": ["Apply Chlorothalonil at first sign",
                     "Use Mancozeb as protective spray",
                     "Spray Iprodione for systemic control"],
        "prevention": "Maintain humidity below 85%, ensure good air circulation, avoid wetting leaves"
    },
    "Septoria leaf spot": {
        "organic":  ["Remove infected lower leaves immediately",
                     "Apply copper spray every 7-10 days",
                     "Mulch soil to prevent spore splash"],
        "chemical": ["Apply Chlorothalonil every 7-10 days",
                     "Use Mancozeb as protective fungicide",
                     "Spray Myclobutanil for systemic control"],
        "prevention": "Crop rotation, remove plant debris after season, avoid overhead watering"
    },
    "Spider mites Two-spotted spider mite": {
        "organic":  ["Spray plants forcefully with water to dislodge mites",
                     "Apply neem oil or insecticidal soap every 5-7 days",
                     "Introduce predatory mites (Phytoseiulus persimilis)"],
        "chemical": ["Apply Abamectin (Agri-Mek) at first sign",
                     "Use Bifenazate (Floramite) for quick knockdown",
                     "Spray Spiromesifen (Oberon) for egg and nymph control"],
        "prevention": "Monitor regularly, avoid over-fertilizing with nitrogen, maintain humidity"
    },
    "Target Spot": {
        "organic":  ["Remove infected leaves and destroy them",
                     "Apply copper-based fungicide weekly",
                     "Improve air circulation around plants"],
        "chemical": ["Apply Chlorothalonil at first sign",
                     "Use Azoxystrobin for systemic protection",
                     "Spray Fluxapyroxad + Pyraclostrobin (Priaxor) for best control"],
        "prevention": "Crop rotation, remove plant debris, avoid overhead irrigation"
    },
    "Tomato Yellow Leaf Curl Virus": {
        "organic":  ["Use reflective silver mulch to repel whiteflies",
                     "Install yellow sticky traps around plants",
                     "Spray neem oil to reduce whitefly population"],
        "chemical": ["Apply Imidacloprid (Confidor) to control whiteflies",
                     "Use Thiamethoxam (Actara) as systemic insecticide",
                     "Spray Spiromesifen (Oberon) for whitefly larvae control"],
        "prevention": "Use virus-resistant tomato varieties, remove infected plants immediately"
    },
    "Tomato mosaic virus": {
        "organic":  ["Remove and destroy infected plants immediately",
                     "Disinfect tools with bleach solution between plants",
                     "Wash hands thoroughly before handling plants"],
        "chemical": ["No chemical cure exists — focus on prevention",
                     "Control aphid vectors with Imidacloprid",
                     "Apply mineral oil spray to reduce virus transmission"],
        "prevention": "Use virus-resistant varieties, certified disease-free seeds, sterilize tools"
    },
    "healthy": {
        "organic":  ["Continue regular compost application",
                     "Use neem oil as preventive spray monthly",
                     "Maintain proper watering schedule"],
        "chemical": [],
        "prevention": "Regular monitoring, balanced fertilization, proper plant spacing"
    }
}

with open("agent/treatment_data.json", "w") as f:
    json.dump(treatment_data, f, indent=2)

# Download it so you have it on laptop too
from google.colab import files
files.download("agent/treatment_data.json")
print(f"treatment_data.json saved with {len(treatment_data)} entries!")
print("Downloaded to your laptop — save it to agent/ folder!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

treatment_data.json saved with 28 entries!
Downloaded to your laptop — save it to agent/ folder!


In [9]:
from google.colab import files
print("Upload disease_data.json and treatment_data.json")
uploaded = files.upload()

import shutil
for filename in uploaded:
    shutil.copy(filename, f"agent/{filename}")
    print(f"Saved: agent/{filename}")

Upload disease_data.json and treatment_data.json


Saving treatment_data.json to treatment_data.json
Saving disease_data.json to disease_data (1).json
Saved: agent/treatment_data.json
Saved: agent/disease_data (1).json


In [3]:
SYSTEM_PROMPT = """You are Dr. Krishi, an expert agricultural plant pathologist with 20 years
of field experience helping farmers across India and Southeast Asia.

Your role:
- Diagnose plant diseases accurately based on ML model predictions
- Give practical, affordable treatment advice farmers can act on immediately
- Speak in a warm, caring tone — farmers may be stressed about losing their crops
- Always mention severity clearly so farmers understand urgency
- Prioritise organic treatments first, then chemical as backup
- End every diagnosis with one key prevention tip for next season

Your response style:
- Clear headers for each section
- Simple language — avoid overly technical jargon
- Specific product names with generic alternatives in brackets
- Realistic about limitations — if confidence is below 80%, suggest getting a second opinion

You have access to a structured disease database and always base your advice on that data.
Never guess or hallucinate treatment names."""

print("System prompt defined!")
print(f"Prompt length: {len(SYSTEM_PROMPT)} characters")

System prompt defined!
Prompt length: 972 characters


In [4]:
def disease_info(disease_name):
    with open("agent/disease_data.json", "r") as f:
        data = json.load(f)
    if disease_name in data:
        info = data[disease_name]
        return {"disease": disease_name, **info, "found": True}
    for key in data:
        if key.lower() in disease_name.lower() or disease_name.lower() in key.lower():
            info = data[key]
            return {"disease": key, **info, "found": True}
    return {"disease": disease_name, "cause": "Unknown",
            "symptoms": "Unknown", "severity": "Unknown", "found": False}

def treatment_advice(disease_name, farming_type="both"):
    with open("agent/treatment_data.json", "r") as f:
        data = json.load(f)
    matched_key = None
    if disease_name in data:
        matched_key = disease_name
    else:
        for key in data:
            if key.lower() in disease_name.lower() or disease_name.lower() in key.lower():
                matched_key = key
                break
    if not matched_key:
        return {"disease": disease_name, "organic": ["Consult local agricultural officer"],
                "chemical": ["Consult local agricultural officer"],
                "prevention": "No data available", "found": False}
    info   = data[matched_key]
    result = {"disease": matched_key, "prevention": info["prevention"], "found": True}
    if farming_type in ("organic", "both"):
        result["organic"]  = info["organic"]
    if farming_type in ("chemical", "both"):
        result["chemical"] = info["chemical"]
    return result

print("Both tools ready!")

Both tools ready!


In [5]:
def run_expert_agent(plant, disease, confidence, farming_type="both"):
    """
    Expert agent with Dr. Krishi persona.
    Takes structured prediction + tool results → expert diagnosis report.
    """

    info      = disease_info(disease)
    treatment = treatment_advice(disease, farming_type)

    # Handle low confidence differently
    confidence_note = ""
    if confidence < 80:
        confidence_note = f"\nNOTE: Model confidence is only {confidence}%. Recommend visual confirmation."
    elif confidence < 95:
        confidence_note = f"\nModel confidence: {confidence}% — good but not certain."
    else:
        confidence_note = f"\nModel confidence: {confidence}% — high confidence diagnosis."

    organic_str  = "\n".join([f"  • {t}" for t in treatment.get("organic", [])])
    chemical_str = "\n".join([f"  • {t}" for t in treatment.get("chemical", [])])

    user_message = f"""
Plant: {plant}
Diagnosed disease: {disease}
{confidence_note}

Disease database results:
- Cause: {info.get('cause', 'Unknown')}
- Symptoms: {info.get('symptoms', 'Unknown')}
- Severity: {info.get('severity', 'Unknown')}

Treatment database results:
Organic options:
{organic_str}

Chemical options:
{chemical_str}

Prevention: {treatment.get('prevention', 'Monitor regularly')}

Farming preference: {farming_type}

Please provide a complete diagnosis report as Dr. Krishi.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message}
        ],
        max_tokens=500
    )

    return response.choices[0].message.content

print("Expert agent ready!")

Expert agent ready!


In [10]:
# Scenario 1 — High confidence, high severity
print("SCENARIO 1: High severity disease, high confidence")
print("="*60)
print(run_expert_agent(
    plant="Tomato",
    disease="Tomato Late blight",
    confidence=99.9,
    farming_type="both"
))

# Scenario 2 — Organic farmer
print("\n\nSCENARIO 2: Organic farmer with medium severity")
print("="*60)
print(run_expert_agent(
    plant="Apple",
    disease="Apple scab",
    confidence=97.5,
    farming_type="organic"
))

# Scenario 3 — Low confidence (edge case)
print("\n\nSCENARIO 3: Low confidence prediction")
print("="*60)
print(run_expert_agent(
    plant="Corn",
    disease="Northern Leaf Blight",
    confidence=71.2,
    farming_type="both"
))

SCENARIO 1: High severity disease, high confidence
## Introduction
I'm Dr. Krishi, and I'm here to help you with your tomato crop. Based on the diagnosis, I can see that your plants are suffering from Tomato Late Blight, a severe disease that requires immediate attention.

## Disease Diagnosis
The model has diagnosed your plants with Tomato Late Blight with a confidence level of 99.9%, which is extremely high. This means we can be almost certain that the diagnosis is accurate. The disease is caused by the water mold Phytophthora infestans, and the symptoms you're seeing, such as greasy grey-green water-soaked patches on the leaves and white mold on the undersides, are consistent with this diagnosis.

## Disease Severity
The severity of the disease is High, which means it's essential to take immediate action to prevent further damage to your crop. If left untreated, Tomato Late Blight can spread quickly and destroy your entire crop.

## Treatment Options
We have both organic and chemica

In [ ]:
agent_code = '''import os
import json
from groq import Groq

SYSTEM_PROMPT = """You are Dr. Krishi, an expert agricultural plant pathologist with 20 years
of field experience helping farmers across India and Southeast Asia.

Your role:
- Diagnose plant diseases accurately based on ML model predictions
- Give practical, affordable treatment advice farmers can act on immediately
- Speak in a warm, caring tone — farmers may be stressed about losing their crops
- Always mention severity clearly so farmers understand urgency
- Prioritise organic treatments first, then chemical as backup
- End every diagnosis with one key prevention tip for next season

Your response style:
- Clear headers for each section
- Simple language — avoid overly technical jargon
- Specific product names with generic alternatives in brackets
- Realistic about limitations — if confidence is below 80%, suggest getting a second opinion

You have access to a structured disease database and always base your advice on that data.
Never guess or hallucinate treatment names."""


def get_client(api_key):
    return Groq(api_key=api_key)


def disease_info(disease_name, data_path="agent/disease_data.json"):
    with open(data_path, "r") as f:
        data = json.load(f)
    if disease_name in data:
        info = data[disease_name]
        return {"disease": disease_name, **info, "found": True}
    for key in data:
        if key.lower() in disease_name.lower() or disease_name.lower() in key.lower():
            info = data[key]
            return {"disease": key, **info, "found": True}
    return {"disease": disease_name, "cause": "Unknown",
            "symptoms": "Unknown", "severity": "Unknown", "found": False}


def treatment_advice(disease_name, farming_type="both",
                     data_path="agent/treatment_data.json"):
    with open(data_path, "r") as f:
        data = json.load(f)
    matched_key = None
    if disease_name in data:
        matched_key = disease_name
    else:
        for key in data:
            if key.lower() in disease_name.lower() or disease_name.lower() in key.lower():
                matched_key = key
                break
    if not matched_key:
        return {"disease": disease_name,
                "organic": ["Consult local agricultural officer"],
                "chemical": ["Consult local agricultural officer"],
                "prevention": "No data available", "found": False}
    info   = data[matched_key]
    result = {"disease": matched_key, "prevention": info["prevention"], "found": True}
    if farming_type in ("organic", "both"):
        result["organic"]  = info["organic"]
    if farming_type in ("chemical", "both"):
        result["chemical"] = info["chemical"]
    return result


def run_expert_agent(plant, disease, confidence, client, farming_type="both"):
    info      = disease_info(disease)
    treatment = treatment_advice(disease, farming_type)

    if confidence < 80:
        confidence_note = f"NOTE: Model confidence is only {confidence}%. Recommend visual confirmation."
    elif confidence < 95:
        confidence_note = f"Model confidence: {confidence}% — good but not certain."
    else:
        confidence_note = f"Model confidence: {confidence}% — high confidence diagnosis."

    organic_str  = "\\n".join([f"  • {t}" for t in treatment.get("organic", [])])
    chemical_str = "\\n".join([f"  • {t}" for t in treatment.get("chemical", [])])

    user_message = f"""
Plant: {plant}
Diagnosed disease: {disease}
{confidence_note}

Disease database results:
- Cause: {info.get("cause", "Unknown")}
- Symptoms: {info.get("symptoms", "Unknown")}
- Severity: {info.get("severity", "Unknown")}

Treatment database results:
Organic options:
{organic_str}

Chemical options:
{chemical_str}

Prevention: {treatment.get("prevention", "Monitor regularly")}

Farming preference: {farming_type}

Please provide a complete diagnosis report as Dr. Krishi.
"""

    response = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message}
        ],
        max_tokens=500
    )

    return response.choices[0].message.content
'''

with open("agent.py", "w") as f:
    f.write(agent_code)

from google.colab import files
files.download("agent.py")
print("agent.py saved and downloaded!")
print("Save this to your agent/ folder on laptop.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

agent.py saved and downloaded!
Save this to your agent/ folder on laptop.
